In [ ]:
# Qwen Unbound Coder AI — setup (run in order)
import os, subprocess, sys
print('Python', sys.version)
os.makedirs('qwen', exist_ok=True)
os.chdir('qwen')
# Repo files: upload this repo's ZIP to the notebook, or clone your fork:
#   !git clone https://github.com/ryoaonetsuki/qwen-unbound-coder-ai.git
# (Replace with YOUR repo URL after you create it — do not use a URL you have not created.)


In [ ]:
!nvidia-smi
# EXPECTED on free Colab: Tesla T4 ~15 GB usable. On Kaggle dual-T4: two T4s.
# If this cell says 'GPU usage limits' / no GPU, STOP here. Do not retry in a loop.


In [ ]:
# Download the EXACT requested file (16.8 GB, first run per session).
# Colab local files do not survive a new runtime. Drive caching is optional and
# only where the provider permits it; this cell downloads to local disk by default.
!python3 ../qwen-unbound-coder-ai/scripts/download_model.py --quant Q4_K_M --dir models 2>&1 | tail -20
!ls -lh models/
# Check the size is ~16.8 GB before starting the server. A partial file will fail to load.


In [ ]:
# llama.cpp binary (CPU build in notebook is enough to fetch a release binary;
# for CUDA, use the official CUDA release asset for your CUDA version — check the
# llama.cpp releases page at run time, do not hard-code a stale URL here.)
# Example (verify the tag first):
#   !curl -L -o llama.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/bXXXX/llama-bXXXX-bin-ubuntu-cuda-12.4-x64.tar.gz
#   !tar xzf llama.tar.gz && ls llama-*/bin/llama-server
print('Download a current llama.cpp release binary, then set LLAMA_BIN below.')
import shutil; print('llama-server on PATH:', shutil.which('llama-server'))


In [ ]:
import os, subprocess, time, threading
os.environ['QWEN_API_KEY'] = 'CHANGE-ME-IN-THIS-CELL-ONLY'  # invent a long random key; never commit it
os.environ['LLAMA_URL'] = 'http://127.0.0.1:8080'
# Kaggle dual T4: tensor-split across 2x16 GB (~30 GB usable) is the free path
# that can hold Q4_K_M fully on GPU at 8k ctx (~18.6 GB total). Verify both GPUs
# in nvidia-smi first; if only one GPU is allocated, use the Colab partial-offload path.
proc1 = subprocess.Popen(['bash','../qwen-unbound-coder-ai/backend/llama_server.sh'], env={**os.environ,'NGL':'99','TENSOR_SPLIT':'0.5,0.5','CTX':'8192'})
proc2 = subprocess.Popen(['python3','../qwen-unbound-coder-ai/backend/gateway.py'], env={**os.environ,'GATEWAY_PORT':'8000'})
time.sleep(10)
import urllib.request, json
print(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=10).read().decode())
print('If backend_reachable is false, read the llama-server log before chatting.')


In [ ]:
# Expose ONLY the gateway (port 8000), never llama-server directly.
# cloudflared quick tunnel needs no account/token for a temporary URL:
!curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
# Run in background and watch its log for the https://....trycloudflare.com URL:
import subprocess; subprocess.Popen(['./cloudflared','tunnel','--url','http://127.0.0.1:8000'])
print('Wait ~20s, then read the cloudflared output for your tunnel URL. Paste it into the frontend.')


In [ ]:
# REAL model test — only after /health says online. This prints the genuine reply.
import os
os.environ['GATEWAY_URL']='http://127.0.0.1:8000'
os.environ['QWEN_API_KEY']='CHANGE-ME-IN-THIS-CELL-ONLY'
!python3 ../qwen-unbound-coder-ai/scripts/test_live.py
